In [10]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    precision_recall_curve
)
from sklearn.calibration import calibration_curve

# Seed policy for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_dir = os.path.join("..", "..", "dataset (tasks 2 and 3)")
train_path = os.path.join(data_dir, "train.csv")
test_path = os.path.join(data_dir, "test.csv")

In [11]:
df_train_full = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

num_cols = [f"integer_feature_{i}" for i in range(1, 14)]
cat_cols = [f"categorical_feature_{i}" for i in range(1, 27)]
target_col = "label"

# Class balance check
pd.DataFrame({
    'Count': df_train_full[target_col].value_counts(),
    'Percentage (%)': df_train_full[target_col].value_counts(normalize=True) * 100
})

,Count,Percentage (%)
label,,
0,38720,96.8
1,1280,3.2


## 2. Preprocessing Pipeline



In [12]:
# 1. Stratified Split (80/20)
train_df, val_df = train_test_split(
    df_train_full, test_size=0.20, random_state=42, stratify=df_train_full[target_col]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# 2. Missingness Indicators
high_missing_cols = ['integer_feature_4', 'integer_feature_5', 'integer_feature_11']
for c in high_missing_cols:
    train_df[f'{c}_isna'] = train_df[c].isnull().astype(np.float32)
    val_df[f'{c}_isna'] = val_df[c].isnull().astype(np.float32)
    test_df[f'{c}_isna'] = test_df[c].isnull().astype(np.float32)

indicator_cols = [f'{c}_isna' for c in high_missing_cols]

# 3. Numerical Imputation (Median) & Log1p
for c in num_cols:
    med = train_df[c].median()
    train_df[c] = train_df[c].fillna(med)
    val_df[c] = val_df[c].fillna(med)
    test_df[c] = test_df[c].fillna(med)
    
    min_val = train_df[c].min()
    train_df[c] = np.log1p(np.maximum(0, train_df[c] - min_val))
    val_df[c] = np.log1p(np.maximum(0, val_df[c] - min_val))
    test_df[c] = np.log1p(np.maximum(0, test_df[c] - min_val))

# Standardize continuous features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(train_df[num_cols])
X_val_cont = scaler.transform(val_df[num_cols])
X_test_cont = scaler.transform(test_df[num_cols])

# Combine standardized features with indicators (13 + 3 = 16 numerical inputs)
X_train_num = np.hstack([X_train_cont, train_df[indicator_cols].values])
X_val_num = np.hstack([X_val_cont, val_df[indicator_cols].values])
X_test_num = np.hstack([X_test_cont, test_df[indicator_cols].values])

# 4. Categorical Vocabularies (Min Frequency = 5, Index 0 = OOV)
MIN_FREQ = 5
cat_modes = {}
cat_vocabs = {}
cat_vocab_sizes = []

for c in cat_cols:
    mode_val = train_df[c].mode()[0]
    cat_modes[c] = mode_val
    
    filled_train = train_df[c].fillna(mode_val)
    counts = filled_train.value_counts()
    frequent = counts[counts >= MIN_FREQ].index.tolist()
    
    vocab = {val: idx + 1 for idx, val in enumerate(frequent)}
    cat_vocabs[c] = vocab
    cat_vocab_sizes.append(len(vocab) + 1)

def encode_cats(df, cols, vocabs, modes):
    encoded = np.zeros((len(df), len(cols)), dtype=np.int64)
    for i, c in enumerate(cols):
        v = vocabs[c]
        m = modes[c]
        series = df[c].fillna(m)
        encoded[:, i] = series.map(lambda x: v.get(x, 0)).values
    return encoded

X_train_cat = encode_cats(train_df, cat_cols, cat_vocabs, cat_modes)
X_val_cat = encode_cats(val_df, cat_cols, cat_vocabs, cat_modes)
X_test_cat = encode_cats(test_df, cat_cols, cat_vocabs, cat_modes)

y_train = train_df[target_col].values.astype(np.float32)
y_val = val_df[target_col].values.astype(np.float32)
y_test = test_df[target_col].values.astype(np.float32)

pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Samples': [len(y_train), len(y_val), len(y_test)],
    'Numerical Features': [X_train_num.shape[1]] * 3,
    'Categorical Features': [X_train_cat.shape[1]] * 3
})

,Split,Samples,Numerical Features,Categorical Features
0,Train,32000,16,26
1,Validation,8000,16,26
2,Test,10000,16,26


## 3. PyTorch Dataset & DataLoader

In [13]:
class CTRDataset(Dataset):
    def __init__(self, num_feats, cat_feats, labels=None):
        self.num_feats = torch.tensor(num_feats, dtype=torch.float32)
        self.cat_feats = torch.tensor(cat_feats, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32) if labels is not None else None

    def __len__(self):
        return len(self.num_feats)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.num_feats[idx], self.cat_feats[idx], self.labels[idx]
        return self.num_feats[idx], self.cat_feats[idx]

BATCH_SIZE = 512

train_loader = DataLoader(CTRDataset(X_train_num, X_train_cat, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CTRDataset(X_val_num, X_val_cat, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(CTRDataset(X_test_num, X_test_cat, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 4. DLRM Architecture Implementation

The architecture strictly follows the specification from the paper and instructions:
1. **Categorical Embeddings**: Each of the 26 categorical features is mapped to an embedding table of uniform vector dimension $d$.
2. **Bottom (Lower) MLP**: Preprocessed numerical features pass through a sequence of `Linear` $\rightarrow$ `ReLU` layers. The output dense vector has length exactly equal to $d$.
3. **Explicit Pairwise Feature Interaction**:
   - We assemble the dense representation vector $v_0 \in \mathbb{R}^d$ and the 26 categorical embedding vectors $v_1, \dots, v_{26} \in \mathbb{R}^d$ into a combined matrix $V \in \mathbb{R}^{27 \times d}$.
   - We compute all pairwise dot products via batched matrix multiplication: $M = V V^T \in \mathbb{R}^{27 \times 27}$.
   - The strictly upper triangular elements are extracted, yielding an interaction vector of dimension:
     $$\binom{27}{2} = \frac{27 \times 26}{2} = 351$$
4. **Top (Output) MLP**:
   - The interaction vector (351) is concatenated with the original dense vector ($d$), forming an input of dimension $d + 351$.
   - The concatenated vector passes through the Top MLP (`Linear` $\rightarrow$ `ReLU` for hidden layers, and `Linear` $\rightarrow$ `Sigmoid` for the final click probability).
5. **Loss Function**: Binary Cross-Entropy (`nn.BCELoss()`).

<p align="center">
  <img src="https://arxiv.org/html/1906.00091v1/net.png" width="500">
</p>

**Figure: DLRM architecture**

In [14]:
import torch
import torch.nn as nn


class DLRM(nn.Module):
    def __init__(
        self,
        embedding_sizes,
        embedding_dim=16,
        num_numerical=16,
        bottom_mlp_dims=[128, 64],
        top_mlp_dims=[256, 128],
        use_interaction=True,
        dropout=0.2,
        ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.use_interaction = use_interaction
        self.num_cats = len(embedding_sizes)

        # 1. Categorical embeddings (all of equal dimension d); index 0 = OOV
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=v, embedding_dim=embedding_dim, padding_idx=0)
            for v in embedding_sizes
        ])

        # 2. Bottom MLP: numerical features -> dense vector of dimension d
        bottom_layers = []
        in_dim = num_numerical
        for h_dim in bottom_mlp_dims:
            bottom_layers.append(nn.Linear(in_dim, h_dim))
            bottom_layers.append(nn.ReLU())
            in_dim = h_dim
        bottom_layers.append(nn.Linear(in_dim, embedding_dim))
        # ReLU on the output only when the MLP has hidden layers; with
        # bottom_mlp_dims=[] (Ablation 4) this is a pure single linear projection
        if len(bottom_mlp_dims) > 0:
            bottom_layers.append(nn.ReLU())
        self.bottom_mlp = nn.Sequential(*bottom_layers)

        # 3. Pairwise interaction setup
        num_vectors = 1 + self.num_cats  # 1 dense vector + 26 embeddings = 27
        if use_interaction:
            num_interactions = (num_vectors * (num_vectors - 1)) // 2  # 351
            triu_indices = torch.triu_indices(num_vectors, num_vectors, offset=1)
            self.register_buffer("triu_row", triu_indices[0])
            self.register_buffer("triu_col", triu_indices[1])
            top_in_dim = embedding_dim + num_interactions
        else:
            # Ablation: concatenate dense vector + all embeddings directly
            top_in_dim = embedding_dim + self.num_cats * embedding_dim

        # 4. Top MLP: Linear -> ReLU -> Dropout, final Linear -> Sigmoid
        top_layers = []
        in_dim = top_in_dim
        for h_dim in top_mlp_dims:
            top_layers.append(nn.Linear(in_dim, h_dim))
            top_layers.append(nn.ReLU())
            top_layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        top_layers.append(nn.Linear(in_dim, 1))
        top_layers.append(nn.Sigmoid())
        self.top_mlp = nn.Sequential(*top_layers)

    def interact(self, dense_vec, emb_vectors):
        # (B, 27, d)
        all_vecs = torch.stack([dense_vec] + emb_vectors, dim=1)
        # (B, 27, 27) all pairwise dot products
        dot_products = torch.bmm(all_vecs, all_vecs.transpose(1, 2))
        # (B, 351) strictly upper-triangular entries
        return dot_products[:, self.triu_row, self.triu_col]

    def forward(self, x_num, x_cat):
        dense_vec = self.bottom_mlp(x_num)
        emb_vectors = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]

        if self.use_interaction:
            interaction_vec = self.interact(dense_vec, emb_vectors)
            concat_vec = torch.cat([dense_vec, interaction_vec], dim=1)
        else:
            concat_vec = torch.cat([dense_vec] + emb_vectors, dim=1)

        return self.top_mlp(concat_vec).squeeze(-1)

## 5. Training & Evaluation Pipeline

- **Loss Function**: `nn.BCELoss()`
- **Optimizer**: Adam (`lr=1e-3`, `weight_decay=1e-5`)
- **Evaluation Metrics**: ROC-AUC, PR-AUC, Log Loss, Accuracy, F1-Score, Precision, and Recall.
- **Model Checkpointing**: Caches the best model weights based on validation ROC-AUC.

In [15]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_preds = []
    
    with torch.no_grad():
        for x_num, x_cat, y in dataloader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            probs = model(x_num, x_cat)
            probs_clamped = torch.clamp(probs, 1e-7, 1.0 - 1e-7)
            loss = criterion(probs_clamped, y)
            total_loss += loss.item() * len(y)
            
            all_preds.extend(probs.cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            
    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    avg_loss = total_loss / len(all_targets)
    
    auc = roc_auc_score(all_targets, all_preds)
    pr_auc = average_precision_score(all_targets, all_preds)
    ll = log_loss(all_targets, np.clip(all_preds, 1e-7, 1 - 1e-7))
    bin_preds = (all_preds >= 0.5).astype(int)
    acc = accuracy_score(all_targets, bin_preds)
    f1 = f1_score(all_targets, bin_preds, zero_division=0)
    
    return {
        "loss": avg_loss,
        "auc": auc,
        "pr_auc": pr_auc,
        "log_loss": ll,
        "accuracy": acc,
        "f1": f1,
        "targets": all_targets,
        "preds": all_preds
    }

def train_model(model, train_loader, val_loader, epochs=8, lr=1e-3, weight_decay=1e-5):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": []}
    best_val_auc = 0.0
    best_state = None
    start_time = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_targets = []
        train_preds = []
        
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            probs = model(x_num, x_cat)
            probs_clamped = torch.clamp(probs, 1e-7, 1.0 - 1e-7)
            loss = criterion(probs_clamped, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * len(y)
            train_preds.extend(probs.detach().cpu().numpy())
            train_targets.extend(y.cpu().numpy())
            
        train_loss /= len(train_targets)
        train_auc = roc_auc_score(train_targets, train_preds)
        val_metrics = evaluate_model(model, val_loader, criterion, device)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_metrics["auc"])
        
        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
    total_time = time.time() - start_time
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return history, total_time

## 6. Training the Standard DLRM Model

We instantiate the standard DLRM model with:
- Categorical embedding dimension $d=16$ (following the Criteo experimental setup in Section 5.1 of the paper)
- Bottom MLP: [16 $\rightarrow$ 128 $\rightarrow$ 64 $\rightarrow$ 16]
- Pairwise dot interactions: $\binom{27}{2} = 351$ dot products
- Top MLP: [367 $\rightarrow$ 256 $\rightarrow$ 128 $\rightarrow$ 1] with Sigmoid output

In [16]:
seed_everything(42)
dlrm_model = DLRM(
    embedding_sizes=cat_vocab_sizes,
    embedding_dim=16,
    num_numerical=X_train_num.shape[1],
    bottom_mlp_dims=[128, 64],
    top_mlp_dims=[256, 128],
    use_interaction=True
).to(device)

criterion = nn.BCELoss()
dlrm_history, dlrm_time = train_model(dlrm_model, train_loader, val_loader, epochs=8, lr=1e-3)
dlrm_val_metrics = evaluate_model(dlrm_model, val_loader, criterion, device)

total_params = sum(p.numel() for p in dlrm_model.parameters() if p.requires_grad)
emb_params = sum(p.numel() for p in dlrm_model.embeddings.parameters())
mlp_params = total_params - emb_params

pd.DataFrame({
    'Metric': [
        'Total Parameters',
        'Embedding Parameters',
        'MLP Parameters (% of total)',
        'Training Time (s)',
        'Validation ROC-AUC',
        'Validation PR-AUC',
        'Validation Log Loss'
    ],
    'Value': [
        f"{total_params:,}",
        f"{emb_params:,}",
        f"{mlp_params:,} ({mlp_params / total_params * 100:.1f}%)",
        f"{dlrm_time:.2f}s",
        f"{dlrm_val_metrics['auc']:.4f}",
        f"{dlrm_val_metrics['pr_auc']:.4f}",
        f"{dlrm_val_metrics['loss']:.4f}"
    ]
})

,Metric,Value
0,Total Parameters,"296,881"
1,Embedding Parameters,"158,176"
2,MLP Parameters (% of total),"138,705 (46.7%)"
3,Training Time (s),25.93s
4,Validation ROC-AUC,0.6822
5,Validation PR-AUC,0.0870
6,Validation Log Loss,0.1355
